# Lakeside heating DSM — hourly demand surrogate

**E+ farm (preferred) → scikit-learn → joblib / Excel / Rust desktop**

| | |
|---|---|
| **Question** | Which regressor best predicts `facility_kw` from weather + 6-Area HP occupancy / preheat knobs? |
| **Validation** | `GroupKFold` by **day** (no same-day leakage) |
| **Metrics** | MAE / RMSE overall + **morning peak HE 05–09** |
| **Data** | Prefer `ENERGYPLUS_SIMULATED` farm parquet (`train_parquet_path`) |
| **Honesty** | IdealLoads+COP farm or BAS proxy · status **CANDIDATE** — not tariff-grade |
| **Ship path** | `ml/artifacts/heating_dsm_hourly_v1.joblib` |
| **Costs** | Editable $/kWh + $/kW via `cost_from_hourly_kw` (same as desktop) |

Helpers: `ml/feature_compile_heating_dsm.py`, `ml/train_heating_dsm.py`, `ml/notebook_plots.py`. Spec: `vibe22_agent_spec/HEATING_DSM.md`.


## 0 · Setup

In [1]:
from pathlib import Path
import sys
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

ROOT = Path("..").resolve()
if not (ROOT / "ml").is_dir():
    ROOT = Path(".").resolve()
ML = ROOT / "ml"
sys.path.insert(0, str(ML))

from artifact_paths import artifact_paths, train_parquet_path
from feature_compile_heating_dsm import (
    FEATURE_COLS, compile_features, matrix_xy, morning_peak_mask,
    assert_no_future_leakage, cost_from_hourly_kw,
)
from train_heating_dsm import bake_off
from notebook_plots import (
    family_cv_mae_bars, family_mae_rmse_grouped, leaderboard_table,
    oat_vs_kw_scatter, strategy_morning_peak_bars, example_day_profiles,
    residual_hist, save_fig,
)

PATHS = artifact_paths()
PATHS["figures"].mkdir(parents=True, exist_ok=True)
print("ROOT", ROOT)
print("features", len(FEATURE_COLS))


ROOT C:\Users\ben\Documents\py-bacnet-stacks-playground\vibe_code_apps_22
features 39


## 1 · Load train parquet (E+ farm preferred)


In [2]:
import subprocess
pq = train_parquet_path()
if not pq.is_file():
    farm_script = ROOT / "scripts" / "eplus_heating_dsm_farm.py"
    if farm_script.is_file():
        subprocess.check_call([sys.executable, "-u", str(farm_script)], cwd=str(ROOT))
        pq = train_parquet_path()
    if not pq.is_file():
        subprocess.check_call([sys.executable, "-u", str(ML / "build_bootstrap_dataset.py")], cwd=str(ROOT))
        pq = train_parquet_path()
df = pd.read_parquet(pq)
src = str(df["provenance"].iloc[0]) if "provenance" in df.columns and len(df) else "unknown"
print("parquet", pq)
print("provenance", src, "shape", df.shape)
print(df["strategy_id"].value_counts())
df.head(3)


parquet C:\Users\ben\Documents\py-bacnet-stacks-playground\vibe_code_apps_22\ml\artifacts\heating_dsm_eplus_farm_hourly.parquet
provenance ENERGYPLUS_SIMULATED shape (2880, 33)
strategy_id
baseline           576
stagger_preheat    576
flat_24_7          576
deep_setback       576
morning_all_on     576
Name: count, dtype: int64


,day,simulation_id,hour_ending,month,doy,is_weekend,occupied,oat_f,rh_pct,ghi,...,occ_frac_2F_B,hp_on_1F_A,hp_on_1F_B,hp_on_1F_C,hp_on_1F_D,hp_on_2F_A,hp_on_2F_B,twin_idf,heat_cop_proxy,schema_version
0,2026-01-23,2026-01-23__baseline,0,1,23,0.0,0.0,-5.641667,59.375000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,lakeside_6zone_gshp_best.idf,3.5,lakeside.heating_dsm_farm.v1
1,2026-01-23,2026-01-23__baseline,1,1,23,0.0,0.0,-7.775000,58.708333,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,lakeside_6zone_gshp_best.idf,3.5,lakeside.heating_dsm_farm.v1
2,2026-01-23,2026-01-23__baseline,2,1,23,0.0,0.0,-10.500000,52.791667,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,lakeside_6zone_gshp_best.idf,3.5,lakeside.heating_dsm_farm.v1


## 2 · Feature compile + leakage guard

In [3]:
feat = compile_features(df)
assert_no_future_leakage(df)
X, y, groups, cols = matrix_xy(df)
peak = morning_peak_mask(df)
print("X", X.shape, "peak hours", int(peak.sum()), "days", pd.Series(groups).nunique())
feat[FEATURE_COLS].describe().T.head(12)

X (2880, 39) peak hours 600 days 24


,count,mean,std,min,25%,50%,75%,max
hour_ending,2880.0,1.150000e+01,6.923389,0.000000,5.750000,1.150000e+01,17.250000,23.000000
sin_hour,2880.0,-1.727014e-17,0.707230,-1.000000,-0.707107,6.123234e-17,0.707107,1.000000
cos_hour,2880.0,-5.427757e-17,0.707230,-1.000000,-0.707107,-6.123234e-17,0.707107,1.000000
month,2880.0,4.291667e+00,4.954640,1.000000,1.000000,1.000000e+00,12.000000,12.000000
doy,2880.0,1.177917e+02,146.266629,1.000000,22.750000,2.850000e+01,338.750000,353.000000
is_weekend,2880.0,3.333333e-01,0.471486,0.000000,0.000000,0.000000e+00,1.000000,1.000000
occupied,2880.0,2.500000e-01,0.433088,0.000000,0.000000,0.000000e+00,0.250000,1.000000
oat_f,2880.0,7.047541e+00,8.746279,-16.975000,0.661458,8.264583e+00,13.304167,25.308333
oat_lag1,2880.0,7.049986e+00,8.766826,-16.975000,0.622917,8.314583e+00,13.304167,25.308333
hdd65,2880.0,5.795246e+01,8.746279,39.691667,51.695833,5.673542e+01,64.338542,81.975000


## 3 · Exploratory figures

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
oat_vs_kw_scatter(df, ax=axes[0])
strategy_morning_peak_bars(df, ax=axes[1])
plt.tight_layout()
save_fig(PATHS["figures"] / "oat_and_strategy_peak.png", fig)
plt.show()

cold = (
    df[df["is_weekend"] < 0.5]
    .groupby("day")["oat_f"].mean()
    .sort_values()
    .index[0]
)
fig, ax = plt.subplots(figsize=(9, 4))
example_day_profiles(df, cold, ax=ax)
save_fig(PATHS["figures"] / "example_cold_day_strategies.png", fig)
plt.show()
print("example day", cold)

example day 2026-01-23


## 4 · Model bake-off (GroupKFold)

In [5]:
result = bake_off(df, n_splits=4, n_iter=12)
lb = leaderboard_table(result["leaderboard"], result["cv"]["persistence"])
display(lb)
print("champion:", result["champion"], "| beat persistence:", result["beat_persistence_peak"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
family_cv_mae_bars(result["leaderboard"], result["cv"]["persistence"]["mae_peak_05_09"], ax=axes[0])
family_mae_rmse_grouped(result["leaderboard"], result["cv"]["persistence"], ax=axes[1])
plt.tight_layout()
save_fig(PATHS["figures"] / "sklearn_leaderboard.png", fig)
plt.show()


,family,mae,rmse,mae_peak_05_09,rmse_peak_05_09
0,extra_trees,13.497806,21.074259,17.868297,25.340418
1,rf,14.228137,22.502839,18.217973,27.268936
2,hgb,14.423967,22.548536,18.541114,28.159139
3,elasticnet,17.715379,27.839920,24.476004,37.735858
4,ridge,18.388782,27.509176,25.341300,36.742678


champion: extra_trees | beat persistence: True


## 5 · Residuals + feature importances

In [6]:
from sklearn.model_selection import GroupKFold
from sklearn.base import clone

model = result["model"]
oof = np.zeros_like(y)
gkf = GroupKFold(n_splits=result["n_splits"])
for tr, te in gkf.split(X, y, groups):
    m = clone(model)
    m.fit(X[tr], y[tr])
    oof[te] = m.predict(X[te])

fig, ax = plt.subplots(figsize=(6, 4))
residual_hist(y, oof, ax=ax)
save_fig(PATHS["figures"] / "oof_residuals.png", fig)
plt.show()
print("OOF MAE", float(np.mean(np.abs(y - oof))),
      "morning peak MAE", float(np.mean(np.abs(y[peak] - oof[peak]))))

if hasattr(model, "feature_importances_"):
    imp = pd.Series(model.feature_importances_, index=cols).sort_values(ascending=False).head(15)
    fig, ax = plt.subplots(figsize=(7, 5))
    imp.iloc[::-1].plot(kind="barh", ax=ax, color="#2a9d8f")
    ax.set_title("Champion feature importances (top 15)")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    save_fig(PATHS["figures"] / "feature_importances.png", fig)
    plt.show()

OOF MAE 13.49780609759093 morning peak MAE 17.86829724049553


## 6 · Midnight 24h forecast demo + cost playground

Replay a cold day as a stand-in for a midnight forecast. Compare strategies under
PLACEHOLDER engineering rates (`$/kWh` + `$/kW`) — same formula as Rust desktop.


In [7]:
demo_day = cold
# PLACEHOLDER engineering defaults (match desktop / HEATING_DSM.md)
rates = {
    "energy_rate_per_kwh": 0.12,
    "demand_rate_per_kw": 15.0,
    "similar_days_per_year": 90.0,
}

rows = []
for sid in ["baseline", "stagger_preheat", "flat_24_7", "morning_all_on"]:
    sub = df[(df["day"] == demo_day) & (df["strategy_id"] == sid)].sort_values("hour_ending")
    Xd, yd, _, _ = matrix_xy(sub)
    pred = model.predict(Xd)
    cost = cost_from_hourly_kw(pred, **rates)
    cost["strategy_id"] = sid
    cost["true_peak"] = float(sub["facility_kw"].max())
    rows.append(cost)

cost_df = pd.DataFrame(rows).set_index("strategy_id")
display(cost_df[["energy_kwh", "peak_kw", "energy_cost", "demand_cost", "total_cost", "annual_total_stub"]])

fig, ax = plt.subplots(figsize=(9, 4))
for sid in cost_df.index:
    sub = df[(df["day"] == demo_day) & (df["strategy_id"] == sid)].sort_values("hour_ending")
    Xd, _, _, _ = matrix_xy(sub)
    ax.plot(sub["hour_ending"], model.predict(Xd), label=sid, lw=1.8)
ce = rates["energy_rate_per_kwh"]
cd = rates["demand_rate_per_kw"]
ax.set_title(f"Model 24h profiles — {demo_day} (${ce}/kWh + ${cd}/kW)")
ax.set_xlabel("Hour local")
ax.set_ylabel("pred facility_kw")
ax.legend(fontsize=8, frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
save_fig(PATHS["figures"] / "forecast_day_cost_profiles.png", fig)
plt.show()


,energy_kwh,peak_kw,energy_cost,demand_cost,total_cost,annual_total_stub
strategy_id,,,,,,
baseline,2235.295663,201.500000,268.235480,3022.500000,3290.735480,60411.193159
stagger_preheat,2339.453097,201.500000,280.734372,3022.500000,3303.234372,61536.093453
flat_24_7,3484.962925,304.316513,418.195551,4564.747698,4982.943249,92414.571969
morning_all_on,2637.426068,201.502837,316.491128,3022.542549,3339.033677,64754.712127


## 7 · Serialize champion


In [8]:
PATHS["joblib"].parent.mkdir(parents=True, exist_ok=True)
joblib.dump(
    {
        "model": result["model"],
        "feature_cols": result["feature_cols"],
        "champion": result["champion"],
        "best_params": result.get("best_params"),
        "schema": "lakeside.heating_dsm_hourly.v1",
    },
    PATHS["joblib"],
)
summary = {
    "champion": result["champion"],
    "beat_persistence_peak": result["beat_persistence_peak"],
    "training_source": src,
    "training_parquet": str(pq),
    "cv": result["cv"],
    "leaderboard": [
        {"family": e["family"], "oof_metrics": e["oof_metrics"]}
        for e in result["leaderboard"]
    ],
}
PATHS["champion_summary"].write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
print("wrote", PATHS["joblib"])
print("wrote", PATHS["champion_summary"])
print("training_source", src)


wrote C:\Users\ben\Documents\py-bacnet-stacks-playground\vibe_code_apps_22\ml\artifacts\heating_dsm_hourly_v1.joblib
wrote C:\Users\ben\Documents\py-bacnet-stacks-playground\vibe_code_apps_22\ml\artifacts\champion_summary.json
training_source ENERGYPLUS_SIMULATED
